# Session 1 — probe, build data, train (batch A)
Measures what the base model knows, freezes the eval, builds the mixes, and trains the curve + the seed replicates. ~8h. This is the only session that creates `data/eval.jsonl` — **commit/keep it**, everything else depends on it.

**Setup (every session):** New Notebook → Accelerator **GPU T4 x2**, Internet
**on**, Add Data → your `refusal-calibration` dataset. Run the pip cell first
(it forces one kernel restart; that's expected), then Run All.

**Carry work between sessions:** Save Version → *Save & Run All* so
`/kaggle/working` persists, **or** download `results.zip`/`probe.zip` at the end
and re-upload them into the dataset. Either way every stage resumes — finished
work is skipped, a killed step continues where it stopped.

In [ ]:
!pip install -q unsloth trl peft datasets transformers accelerate pyyaml

Bootstrap: put the uploaded repo in the working dir and on the path. **Run this before the rest.**

In [ ]:
import shutil, os, sys, glob
dst = '/kaggle/working/repo'
if os.path.exists(os.path.join(dst, 'tests.py')):
    # a good copy already here (e.g. mid-session rerun) — keep it so any trained
    # adapters / generated data under runs/ survive
    print('reusing existing', dst)
else:
    if os.path.exists(dst):
        shutil.rmtree(dst)      # broken/partial copy from an earlier attempt
    hits = glob.glob('/kaggle/input/**/stages.py', recursive=True)
    assert hits, 'stages.py not found under /kaggle/input — add your dataset as Input'
    src = os.path.dirname(hits[0])
    print('copying repo from', src)
    shutil.copytree(src, dst)
os.chdir(dst)
for p in (dst, os.path.join(dst, 'data')):
    if p not in sys.path:
        sys.path.insert(0, p)
print('cwd:', os.getcwd(), '| has tests.py:', os.path.exists('tests.py'))

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'tests.py'], check=True)  # logic intact first

In [ ]:
from stages import Ctx, probe, build, train
ctx = Ctx()
probe(ctx)      # GPU, ~45 min (skipped if eval already built)
build(ctx)      # CPU, seconds — freezes data/eval.jsonl + mixes

Check the class split printed above: a healthy probe is ~30-60% answerable, ~20-40% unknown. 95% either way = prompt/threshold problem (RUNBOOK.md), not the model. Fix before training on it.

In [ ]:
RUNS_A = ['v3_mix50', 'v2_mix25', 'v4_mix75', 'v1_mix10', 'v5_mix90', 'v12_seed1', 'v13_seed2']
train(ctx, RUNS_A)   # ~1h each; skips any already trained, resumes after a dead session

## Save results

In [ ]:
# results.zip = everything the write-up needs (small). adapters.zip = the LoRA
# weights (~75MB+ each) — download only the versions you'll publish.
!cd /kaggle/working/repo && zip -qr /kaggle/working/results.zip data/eval.jsonl data/eval.lock \
    data/labeled.jsonl data/mix_*.jsonl runs/*/adapter/adapter_config.json 2>/dev/null
!cd /kaggle/working/repo && zip -qr /kaggle/working/adapters.zip runs/*/adapter 2>/dev/null
!cd /kaggle/working/repo && zip -qr /kaggle/working/probe.zip data/probe_raw.jsonl 2>/dev/null
!ls -lh /kaggle/working/*.zip